# IR-TrueFISP (bSSFP): simulation, reconstruction and T1/T2 mapping

Pipeline: (1) build an inversion-recovery TrueFISP sequence with a golden-angle
radial readout, (2) simulate the multi-coil signal on a numerical phantom,
(3) reconstruct a time series of IR frames by view-sharing (butterfly) CG and by
self-supervised DeepADMM, (4) fit $T_1^*$, $T_1$, $T_2$ and proton density.

## Imports and device

All tensors are created on the GPU by default.

In [ ]:
%matplotlib widget

from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
import torch.nn as nn
from tqdm import tqdm

from Modules import default_system
from Modules.CoilSens import cyclic_head_coil
from Modules.fast_iso import isochromat_sim_fast
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import IdentityOperator, LinearOperator
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.Regularizer import Regularizer

torch.set_default_device("cuda")

## Linear operators and CG solver

Complex tensors are handled as flattened real vectors (`view_as_real`), so the
operators act on real vectors of twice the length.

`NonuniformFourierTransformOperator` maps multi-coil image series to non-Cartesian
k-space samples (one trajectory per frame).

In [ ]:
class NonuniformFourierTransformOperator(LinearOperator):
    """Non-Cartesian Fourier operator: multi-coil image series to k-space samples.

    Parameters
    ----------
    k : torch.Tensor
        k-space trajectory of shape ``(n_dims, n_samples, n_frames)``.
    input_shape : Tuple[int, ...]
        Shape ``(n_channels, n_x, n_y, n_frames)`` of the multi-coil image series.
    device : Optional[torch.device], optional
        Device on which the NUFFT objects are placed.
    """

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        _, n_samples, n_frames = k.shape
        n_channels, n_x, n_y, _ = input_shape

        self.n_modes = (n_x, n_y)
        self.k = k
        self.forward_shape = (n_channels, n_x, n_y, 1, n_frames)
        self.adjoint_shape = (n_channels, n_samples, n_frames)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the forward NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil image series.

        Returns
        -------
        torch.Tensor
            Real-valued flattened k-space samples.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened k-space samples.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil image series.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

`ChannelOperator` maps a single image series to multi-coil images (and back, via
the coil-combining adjoint).

In [ ]:
class ChannelOperator(LinearOperator):
    """Coil sensitivity operator: image series to multi-coil image series.

    Parameters
    ----------
    coil_sensitivities : torch.Tensor
        Complex coil sensitivities of shape ``(n_channels, *data_shape)``.
    data_shape : Tuple[int, ...]
        Shape of the image (series).
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        n_channels = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (n_channels, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the operator: image series to multi-coil image series.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image series.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil image series.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint: multi-coil series to coil-combined series.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil image series.

        Returns
        -------
        torch.Tensor
            Real-valued flattened coil-combined image series.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

Conjugate-gradient solver for $Q x = b$ with a linear operator $Q$.

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:
    """Solve ``Q x = b`` with the conjugate-gradient method.

    Parameters
    ----------
    Q : LinearOperator
        Symmetric positive (semi-)definite operator.
    b : torch.Tensor
        Right-hand side.
    x : torch.Tensor
        Initial estimate.
    maxiter : int, optional
        Number of CG iterations.
    disable_progressbar : bool, optional
        Disable the progress bar.

    Returns
    -------
    torch.Tensor
        Estimate of the solution after ``maxiter`` iterations.
    """
    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for _ in tqdm(range(maxiter), disable=disable_progressbar):
        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## Sequence design

Inversion pulse followed by a long train of TrueFISP (bSSFP) readouts with
alternating RF phase (180 degrees) and golden-angle radial spokes. The first
`n_dummy` readouts of the first segment are dummy shots at angle 0 (no ADC).

In [ ]:
def truefisp_seq(
    fov: float = 200e-3,
    n_x: int = 128,
    flip_angle_deg: float = 60.0,
    slice_thickness: float = 5e-3,
    rf_duration: float = 0.8e-3,
    time_bandwidth_product: float = 2.0,
    bandwidth_per_pixel: float = 890.0,
    prephaser_duration: float = 0.6e-3,
    repetition_time: float = 3.12e-3,
    n_readout: int = 1800,
    n_spokes: int = 1,
    undersampling_factor: int = 1,
    n_dummy: int = 0,
    first_inversion_time: float = 20e-3,
    inter_readout_delay: float = 0.0,
    recovery_delay: float = 2000e-3,
    write_seq: bool = False,
    plot: bool = False,
) -> Tuple[pp.Sequence, float, float]:
    """Build the IR-TrueFISP sequence with golden-angle radial readouts.

    Parameters
    ----------
    fov : float, optional
        Field of view in m.
    n_x : int, optional
        Number of samples per spoke.
    flip_angle_deg : float, optional
        Excitation flip angle in degrees.
    slice_thickness : float, optional
        Slice thickness in m.
    rf_duration : float, optional
        Excitation pulse duration in s.
    time_bandwidth_product : float, optional
        Currently unused; the excitation pulse uses a fixed time-bandwidth
        product of 4.
    bandwidth_per_pixel : float, optional
        Readout bandwidth in Hz/pixel.
    prephaser_duration : float, optional
        Duration of the prephaser/rewinder gradients in s.
    repetition_time : float, optional
        Requested repetition time in s (the achieved TR is returned as the
        duration of one GRE block).
    n_readout : int, optional
        Number of readouts after each inversion pulse.
    n_spokes : int, optional
        Total number of inversion segments before undersampling.
    undersampling_factor : int, optional
        Only ``n_spokes // undersampling_factor`` segments are played.
    n_dummy : int, optional
        Number of dummy readouts (no ADC) at the start of the first segment.
    first_inversion_time : float, optional
        Time from the inversion pulse centre to the first excitation in s.
    inter_readout_delay : float, optional
        Additional delay between consecutive readouts in s.
    recovery_delay : float, optional
        Delay after each readout train in s.
    write_seq : bool, optional
        Write the sequence to ``IR_TrueFISP_bssfp.seq``.
    plot : bool, optional
        Plot a short excerpt of the sequence.

    Returns
    -------
    seq : pp.Sequence
        The assembled sequence.
    gre_block_dur : float
        Duration of one GRE block (the achieved TR) in s.
    first_inversion_time : float
        Time from the inversion pulse centre to the first excitation in s.
    """
    system = default_system()
    seq = pp.Sequence(system)
    delta_k = 1 / fov
    grad_raster = seq.grad_raster_time

    rf_inversion, grad_slice_inversion, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(180),
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, grad_slice, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=np.deg2rad(flip_angle_deg),
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
        use="excitation",
    )

    # Readout flat time: a multiple of the gradient raster that is also a multiple
    # of the ADC dwell time of one spoke.
    adc_step = n_x * seq.adc_raster_time
    raster_multiple = 1
    while (
        abs(
            raster_multiple * grad_raster
            - round(raster_multiple * grad_raster / adc_step) * adc_step
        )
        > 1e-9
    ):
        raster_multiple += 1
    readout_block = raster_multiple * grad_raster
    readout_flat = (
        int(round((1.0 / bandwidth_per_pixel) / readout_block)) * readout_block
    )

    grad_readout_base = pp.make_trapezoid(
        channel="x", flat_area=n_x * delta_k, flat_time=readout_flat, system=system
    )
    adc = pp.make_adc(
        num_samples=n_x,
        duration=grad_readout_base.flat_time,
        delay=grad_readout_base.rise_time,
        system=system,
    )

    prephaser_duration = int(np.ceil(prephaser_duration / grad_raster)) * grad_raster
    grad_prephase_base = pp.make_trapezoid(
        channel="x",
        area=-grad_readout_base.area / 2,
        duration=prephaser_duration,
        system=system,
    )
    grad_slice_rephase = pp.make_trapezoid(
        channel="z",
        area=-grad_slice.area / 2,
        duration=prephaser_duration,
        system=system,
    )

    fixed_duration = (
        rf_duration
        + 2 * prephaser_duration
        + grad_readout_base.flat_time
        + grad_readout_base.rise_time
        + grad_readout_base.fall_time
        + 2 * (grad_prephase_base.rise_time + grad_prephase_base.fall_time)
    )
    delay_te = (
        int(np.ceil(max(1e-4, repetition_time - fixed_duration) / grad_raster))
        * grad_raster
    )
    inversion_duration = pp.calc_duration(rf_inversion, grad_slice_inversion)
    delay_ti_start = (
        np.ceil(
            (
                first_inversion_time
                - inversion_duration / 2
                - pp.calc_duration(grad_slice) / 2
            )
            / grad_raster
        )
        * grad_raster
    )
    assert delay_ti_start > 0, "first_inversion_time too short"

    gre_block_dur = (
        pp.calc_duration(grad_prephase_base)
        + pp.calc_duration(grad_slice)
        + delay_te
        + pp.calc_duration(grad_readout_base)  # readout
        + pp.calc_duration(grad_prephase_base)  # in-plane rewind
    )
    delay_inter_readout = np.ceil(inter_readout_delay / grad_raster) * grad_raster

    golden_angle_deg = 111.246
    rf_phase_deg = 0.0
    rf_phase_increment_deg = 180.0
    spoke_counter = 0

    for segment in range(n_spokes // undersampling_factor):
        seq.add_block(rf_inversion, grad_slice_inversion)
        seq.add_block(pp.make_delay(delay_ti_start))

        n_readouts_segment = n_readout + (n_dummy if segment == 0 else 0)
        for readout_idx in range(n_readouts_segment):
            is_dummy = (segment == 0) and (readout_idx < n_dummy)

            rf.phase_offset = rf_phase_deg / 180 * np.pi
            adc.phase_offset = rf_phase_deg / 180 * np.pi
            rf_phase_deg = (rf_phase_deg + rf_phase_increment_deg) % 360

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * golden_angle_deg)
                spoke_counter += 1

            dir_x = np.cos(angle)
            dir_y = np.sin(angle)

            grad_prephase_x = pp.make_trapezoid(
                channel="x",
                area=grad_prephase_base.area * dir_x,
                duration=prephaser_duration,
                system=system,
            )
            grad_prephase_y = pp.make_trapezoid(
                channel="y",
                area=grad_prephase_base.area * dir_y,
                duration=prephaser_duration,
                system=system,
            )

            grad_rewind_x = pp.make_trapezoid(
                channel="x",
                area=grad_prephase_base.area * dir_x,
                duration=prephaser_duration,
                system=system,
            )
            grad_rewind_y = pp.make_trapezoid(
                channel="y",
                area=grad_prephase_base.area * dir_y,
                duration=prephaser_duration,
                system=system,
            )

            seq.add_block(rf, grad_slice)
            seq.add_block(grad_prephase_x, grad_prephase_y, grad_slice_rephase)
            seq.add_block(pp.make_delay(delay_te))

            grad_readout_x = pp.make_trapezoid(
                channel="x",
                amplitude=grad_readout_base.amplitude * dir_x,
                flat_time=grad_readout_base.flat_time,
                rise_time=grad_readout_base.rise_time,
                fall_time=grad_readout_base.fall_time,
                system=system,
            )
            grad_readout_y = pp.make_trapezoid(
                channel="y",
                amplitude=grad_readout_base.amplitude * dir_y,
                flat_time=grad_readout_base.flat_time,
                rise_time=grad_readout_base.rise_time,
                fall_time=grad_readout_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(grad_readout_x, grad_readout_y)
            else:
                seq.add_block(grad_readout_x, grad_readout_y, adc)

            seq.add_block(grad_rewind_x, grad_rewind_y, grad_slice_rephase)

            if delay_inter_readout > 0 and readout_idx < n_readouts_segment - 1:
                seq.add_block(pp.make_delay(delay_inter_readout))

        seq.add_block(pp.make_delay(recovery_delay))

    timing_ok, error_report = seq.check_timing()
    if timing_ok:
        print("2-D TrueFISP timing check passed")
        print(f"  GRE block dur (TR) : {gre_block_dur * 1000:.2f} ms")
        print(f"  Readout train      : {n_readout * gre_block_dur * 1000:.0f} ms")
        scan_time_est = n_spokes * (
            inversion_duration
            + delay_ti_start
            + n_readout * gre_block_dur
            + recovery_delay
        )
        print(f"  Scan time est       : {scan_time_est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {error_report}")

    seq.set_definition("FOV", [fov, fov, slice_thickness])
    seq.set_definition("Name", "IR_TrueFISP_bssfp.seq")
    if write_seq:
        seq.write("IR_TrueFISP_bssfp.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.225))
    return seq, gre_block_dur, first_inversion_time

## Parameters

In [ ]:
device = torch.device("cuda")
n_x = 128
n_channels = 8
fov = 200e-3
slice_thickness = 5e-3
flip_angle_deg = 60.0  # excitation flip angle
rf_duration = 0.8e-3  # RF pulse duration (rf_duration / repetition_time = 20%)
time_bandwidth_product = 2.0  # sinc time-bandwidth product
bandwidth_per_pixel = 890.0  # readout bandwidth (Hz/px)
repetition_time = 3.12e-3  # requested repetition time
prephaser_duration = 0.6e-3  # pre-scan / rewind gradient duration
n_readout = 1800
n_spokes = 1
undersampling_factor = 1
n_dummy = 0  # sample right after the inversion
first_inversion_time = 20e-3
inter_readout_delay = 0.0
recovery_delay = 2000e-3

## Build the sequence

In [ ]:
seq, gre_block_dur, first_inversion_time = truefisp_seq(
    fov=fov,
    n_x=n_x,
    flip_angle_deg=flip_angle_deg,
    slice_thickness=slice_thickness,
    rf_duration=rf_duration,
    time_bandwidth_product=time_bandwidth_product,
    bandwidth_per_pixel=bandwidth_per_pixel,
    prephaser_duration=prephaser_duration,
    repetition_time=repetition_time,
    n_readout=n_readout,
    n_spokes=n_spokes,
    undersampling_factor=undersampling_factor,
    n_dummy=n_dummy,
    first_inversion_time=first_inversion_time,
    inter_readout_delay=inter_readout_delay,
    recovery_delay=recovery_delay,
    write_seq=True,
    plot=False,
)

## Phantom and coil sensitivities

The simulation uses coil sensitivities scaled to the physical FOV; the
reconstruction uses unscaled (and flipped) sensitivities on the image grid.
B0 is set to zero and B1 to one.

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_x))

coil_sensitivities_recon = cyclic_head_coil((n_channels, n_x, n_x, 1)).to(
    torch.complex64
)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_x, 1),
    fov_scaling=(phantom.size[0] / fov, phantom.size[1] / fov, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)

voxel_data = phantom.build()

## Signal simulation

Ground-truth time-domain spin simulation of the sequence written above. The
signal is saved to disk with shape `(n_channels, n_samples)`.

In [ ]:
mr0_seq = mr0.Sequence.import_file("IR_TrueFISP_bssfp.seq")
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr0_seq,
    voxel_data.cuda(),
    spin_count=256,
    spin_dist="rand",
    print_progress=True,
)
signal = signal.permute((1, 0)).contiguous()
torch.save(signal.cpu(), "IR_trueFISP_bssfp_signal.pt")

## k-space trajectory

Read the ADC trajectory from the sequence and normalise it to $[-0.5, 0.5]$.

In [ ]:
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

## Butterfly view-sharing

For every IR frame, radial sample $i$ is taken from a window of neighbouring
spokes whose size is the next Fibonacci number above `n_max * radius_i` (at least
`n_center`): the k-space centre is sampled by few spokes (high temporal
resolution), the periphery by many. Frames advance by `frame_advance` spokes.

In [ ]:
n_center = 8  # projections for the k-space center
n_max = 55  # outermost Fibonacci ring; also the frame time window
# Filter advance per frame in spokes (paper: 8 -> 32 ms; 4 -> 16 ms for finer
# temporal resolution of the IR evolution).
frame_advance = 4
# Not needed: the iterative recon models the exact sampling (the DCF would only
# correct a gridding-only reconstruction).
use_dcf = False

fibonacci_numbers = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]


def next_fibonacci(x: float, numbers: list[int]) -> int:
    """Return the smallest entry of ``numbers`` that is not below ``x``.

    Parameters
    ----------
    x : float
        Lower bound.
    numbers : list[int]
        Ascending list of candidate numbers.

    Returns
    -------
    int
        Smallest candidate greater than or equal to ``x``.
    """
    return next(f for f in numbers if f >= x)


n_samples_total = k_traj_adc.shape[1]
assert n_samples_total % n_x == 0, "trajectory is not a whole number of spokes"
n_spokes_total = n_samples_total // n_x
assert n_spokes_total > n_max, "need more spokes than n_max for view-sharing"

# Radius of each radial sample (0 at center -> 1 at the edge) and its ring count.
radius = np.linspace(0.0, 1.0, n_x)
n_projections = np.array(
    [
        next_fibonacci(max(n_center, int(np.ceil(n_max * radius_i))), fibonacci_numbers)
        for radius_i in radius
    ]
)
# Ram-Lak 1/r DCF (center clamped to the first non-zero radius).
density_comp = np.where(radius > 0, 1.0 / radius, 1.0 / radius[1])

# Butterfly offsets: for sample i, the n_projections[i] projections centered on
# the frame.
offsets_list, radial_index_list = [], []
for i in range(n_x):
    n_proj = int(n_projections[i])
    first_offset = -(n_proj // 2)
    offsets_list.extend(range(first_offset, first_offset + n_proj))
    radial_index_list.extend([i] * n_proj)
proj_offset = np.asarray(offsets_list, dtype=np.int64)  # relative projection offset
radial_index = np.asarray(radial_index_list, dtype=np.int64)  # radial sample index
n_points = proj_offset.size
point_weights = density_comp[radial_index] if use_dcf else np.ones(n_points)

# Valid frames: the outermost window [n - n_max//2, n + n_max//2] must fit.
first_center = n_max // 2
n_frames = (n_spokes_total - n_max - 1) // frame_advance + 1
centers = first_center + frame_advance * np.arange(n_frames)

# Per-frame flat k-space sample indices for the butterfly: (n_frames, n_points).
centers_col = centers[:, None]
proj_offset_row = proj_offset[None, :]
radial_index_row = radial_index[None, :]
spoke_index = centers_col + proj_offset_row
sample_indices = spoke_index * n_x + radial_index_row

# Butterfly gather on the CPU (robust for the complex signal), then move the
# result back to the accelerator. The flat index is frame-major (frame, point).
signal_cpu = signal.cpu().numpy()  # (n_channels, n_samples) complex64
k_traj_cpu = k_traj_adc.cpu().numpy()  # (2, n_samples) float32
flat_indices = sample_indices.reshape(-1)  # (n_frames * n_points,)

k_traj_gathered = k_traj_cpu[:, flat_indices].reshape(2, n_frames, n_points)
k_traj_gathered = k_traj_gathered.transpose(0, 2, 1).copy()
k_traj_adc_reshaped = (
    torch.from_numpy(k_traj_gathered).float().to(k_traj_adc.device)
)  # (2, n_points, n_frames)

signal_gathered = signal_cpu[:, flat_indices].reshape(n_channels, n_frames, n_points)
signal_gathered = signal_gathered.transpose(0, 2, 1).copy()
weights = point_weights[None, :, None]
data_weighted = signal_gathered * weights
data_reshaped = torch.from_numpy(data_weighted.astype(np.complex64)).to(
    signal.device
)  # (n_channels, n_points, n_frames)

coil_sensitivities_recon_tiled = torch.tile(
    coil_sensitivities_recon[:, :, :, None], (1, 1, 1, n_frames)
)

## CG reconstruction of the IR series

Notation: $F$ is the NUFFT, $C$ the coil sensitivities, $A = F C$ the encoding
operator, $Q = A^H A$ and $b = A^H d$. We solve $Q x = b$ with 30 CG iterations
starting from zero.

In [ ]:
F = NonuniformFourierTransformOperator(
    k_traj_adc_reshaped, (n_channels, n_x, n_x, n_frames)
)
C = ChannelOperator(coil_sensitivities_recon_tiled, (n_x, n_x, n_frames))
A = F @ C
b = A.H @ data_reshaped.contiguous().view(torch.float32).ravel()

Q = A.H @ A
x_init = torch.zeros(n_x * n_x * n_frames, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, b, x_init, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_x, n_frames))

## Magnitude and signed series

Frame times are the window centres times the achieved TR. The signed series is
obtained by rotating every pixel by the phase of its last (steady-state) frame;
the brain mask thresholds the steady-state magnitude at 10% of its maximum.

In [ ]:
ximg = x_recon_image.cpu().numpy()  # (n_x, n_x, n_frames) complex
t_frames = centers * gre_block_dur * 1e3  # ms
steady_phase = np.angle(ximg[:, :, -1])
phase_correction = np.exp(-1j * steady_phase)[:, :, None]
signed = np.real(ximg * phase_correction)
mag = np.abs(ximg)

ref = mag[:, :, -1]  # steady-state magnitude, same frame as reconstruction
thresh = 0.1 * ref.max()
brain = ref > thresh

## IR magnitude series

Selected frames of the CG reconstruction (moving average over 5 frames); the
darkest frame (the null point) is highlighted in red.

In [ ]:
def smooth_frame(magnitude: np.ndarray, index: int, half_width: int = 2) -> np.ndarray:
    """Average the magnitude series over a window of frames around ``index``.

    Parameters
    ----------
    magnitude : np.ndarray
        Magnitude series of shape ``(n_x, n_y, n_frames)``.
    index : int
        Centre frame of the window.
    half_width : int, optional
        Number of frames included on each side of ``index``.

    Returns
    -------
    np.ndarray
        Averaged image of shape ``(n_x, n_y)``.
    """
    lo = max(0, index - half_width)
    hi = min(magnitude.shape[-1], index + half_width + 1)
    return magnitude[:, :, lo:hi].mean(-1)


vmax = 1.5 * float(np.percentile(mag, 95))

null_frame = int(np.argmin(mag.reshape(-1, n_frames)[brain.reshape(-1)].mean(0)))
last_frame = min(
    n_frames - 1,
    int((1200.0 - t_frames[0]) / (frame_advance * repetition_time * 1e3)),
)
frame_idx = np.sort(
    np.unique(
        np.concatenate(
            [np.linspace(0, last_frame, 11).round().astype(int), [null_frame]]
        )
    )
)
nrows, ncols = 3, (len(frame_idx) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle("IR magnitude series (grayscale) - darkest frame = null")
for panel, frame in enumerate(frame_idx):
    ax = plt.subplot(nrows, ncols, panel + 1)
    is_null = frame == null_frame
    plt.imshow(smooth_frame(mag, frame).T, cmap="gray", vmin=0, vmax=vmax)
    plt.title(
        f"{t_frames[frame]:.0f} ms",
        fontsize=9,
        color="red" if is_null else "k",
        fontweight="bold" if is_null else "normal",
    )
    for spine in ax.spines.values():
        spine.set_edgecolor("red" if is_null else "0.8")
        spine.set_linewidth(2.5 if is_null else 0.5)
    plt.axis("off")
for panel in range(len(frame_idx), nrows * ncols):
    plt.subplot(nrows, ncols, panel + 1)
    plt.axis("off")
plt.show()

## DeepADMM

Solves $\min_x \tfrac{1}{2}\|Ax - d\|_2^2 + R(x)$ by ADMM, where the $x$-update is
a few CG iterations on $(Q + \rho I)\,x = b + \rho\,(z - u)$ and the $z$-update
applies the learned regularizer. $\rho$ is a trainable parameter.

In [ ]:
class DeepADMM(nn.Module):
    """ADMM solver with a learned regularizer and a trainable penalty weight.

    Parameters
    ----------
    shape : Tuple[int, ...]
        Shape of the image the regularizer and identity operator act on.
    rho_init : float, optional
        Initial value of the ADMM penalty parameter ``rho``.
    maxiter_admm : int, optional
        Number of ADMM iterations.
    maxiter_cg : int, optional
        Number of CG iterations per ADMM ``x``-update.
    """

    def __init__(
        self,
        shape: Tuple[int, ...],
        rho_init: float = 0.1,
        maxiter_admm: int = 10,
        maxiter_cg: int = 5,
    ) -> None:
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(shape)
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg

    def forward(
        self,
        Q: LinearOperator,
        b: torch.Tensor,
        x: torch.Tensor,
        disable_progressbar: bool = True,
    ) -> torch.Tensor:
        """Run the unrolled ADMM iterations.

        Parameters
        ----------
        Q : LinearOperator
            Normal operator ``A^H A``.
        b : torch.Tensor
            Right-hand side ``A^H d`` (real-valued view of the complex image).
        x : torch.Tensor
            Initial image estimate (real-valued view).
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        torch.Tensor
            Reconstructed image (real-valued view).
        """
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)

        for _ in range(self.maxiter_admm):
            x = conjugate_gradient(
                Q + self.rho * self.I,
                b=b + self.rho * (z - u),
                x=x,
                maxiter=self.maxiter_cg,
                disable_progressbar=disable_progressbar,
            )
            z = self.regularizer(x + u)
            u = u + x - z

        return x

## Self-supervised training

Each step picks one of the selected frames and randomly splits its butterfly
points into two halves: the model reconstructs from the source half, and the
loss (relative L1) compares the predicted k-space on the target half with the
measured data.

In [ ]:
shape = (n_x, n_x, 1)
torch.manual_seed(0)
model = DeepADMM(shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=15)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

n_points_per_frame = k_traj_adc_reshaped.shape[1]
C_frame = ChannelOperator(coil_sensitivities_recon, (n_x, n_x))
num_iters = 100

model.train()
losses = []
progress_bar = tqdm(range(num_iters), desc="Self-supervised splits")
for it in progress_bar:
    np.random.seed(it)
    frame = frame_idx[np.random.randint(frame_idx.size)]

    indices = np.arange(n_points_per_frame)
    np.random.shuffle(indices)
    index_src, index_tar = np.array_split(indices, 2)

    k_traj_src = k_traj_adc_reshaped[:, index_src, frame][:, :, None]
    k_traj_tar = k_traj_adc_reshaped[:, index_tar, frame][:, :, None]
    k_data_src = (
        data_reshaped[:, index_src, frame].contiguous().view(torch.float32).ravel()
    )
    k_data_tar = (
        data_reshaped[:, index_tar, frame].contiguous().view(torch.float32).ravel()
    )

    F_src = NonuniformFourierTransformOperator(k_traj_src, (n_channels, n_x, n_x, 1))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (n_channels, n_x, n_x, 1))

    Q_src = C_frame.H @ F_src.H @ F_src @ C_frame
    b_src = C_frame.H @ F_src.H @ k_data_src
    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)
    k_data_pred = F_tar @ C_frame @ x_out

    target_norm = torch.linalg.norm(k_data_tar, ord=1)
    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / target_norm
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

## Trained reconstruction

Reconstruct each selected frame separately with the trained model.

In [ ]:
n_frames_sel = frame_idx.size
k_traj_sel = k_traj_adc_reshaped[:, :, frame_idx]
data_sel = data_reshaped[:, :, frame_idx]

model.eval()
recon = []
for frame in range(n_frames_sel):
    k_traj_frame = k_traj_sel[:, :, frame][:, :, None]
    k_data_frame = data_sel[:, :, frame]
    F_frame = NonuniformFourierTransformOperator(
        k_traj_frame, (n_channels, n_x, n_x, 1)
    )
    A_frame = F_frame @ C_frame
    Q_frame = A_frame.H @ A_frame
    b_frame = A_frame.H @ k_data_frame.contiguous().view(torch.float32).ravel()
    with torch.no_grad():
        x_frame = model(Q_frame, b_frame, b_frame)
    recon.append(x_frame.view(torch.complex64).reshape(n_x, n_x))

x_deep_admm_trained_image = torch.stack(recon, dim=-1)  # (n_x, n_x, n_frames_sel)

In [ ]:
t_frames_sel = t_frames[frame_idx]
vmax = 1.5 * float(np.percentile(np.abs(x_deep_admm_trained_image.cpu().numpy()), 95))

fig, axes = plt.subplots(3, 4, figsize=(7.5, 7.5))
for ax, i in zip(axes.ravel(), range(n_frames_sel)):
    ax.imshow(
        x_deep_admm_trained_image[:, :, i].abs().cpu().numpy().T,
        cmap="gray",
        vmin=0,
        vmax=vmax,
    )
    ax.set_title(f"{t_frames_sel[i]:.0f} ms", fontsize=9)
    ax.axis("off")
plt.suptitle("DeepADMM (learned, trained) - 12 frames")
plt.show()

## $T_1^*$ fit (variable projection)

Each voxel's signed series is modelled as $y(t) = c_1 + c_2\,e^{-t/T_1^*}$ with
$t$ measured from the inversion pulse. For a given $T_1^*$ the linear coefficients
$c_1, c_2$ follow in closed form (least squares); $T_1^*$ is found by a coarse
log-spaced grid search followed by golden-section refinement in $\log T_1^*$.
The steady-state signal is $S_{ss} = c_1$ and the equilibrium signal is
$S_0 = -c_2 - c_1$.

In [ ]:
time_ms = first_inversion_time * 1e3 + t_frames
time_s = torch.tensor(time_ms / 1e3, dtype=torch.float64, device=device)

signed_series = torch.tensor(signed, dtype=torch.float64, device=device)
n_voxels = n_x * n_x
signal_flat = signed_series.reshape(n_voxels, n_frames)
ones = torch.ones(n_frames, dtype=torch.float64, device=device)
time_row = time_s[None, :]
ones_row = ones[None, :]


def closed_form_sse(
    t1_star: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Solve for ``c1`` and ``c2`` in closed form and return the fit error.

    Parameters
    ----------
    t1_star : torch.Tensor
        Trial values of ``T1*`` in s, shape ``(n_voxels,)``.

    Returns
    -------
    sse : torch.Tensor
        Sum of squared residuals per voxel, shape ``(n_voxels,)``.
    c1 : torch.Tensor
        Constant term per voxel, shape ``(n_voxels,)``.
    c2 : torch.Tensor
        Amplitude of the decay term per voxel, shape ``(n_voxels,)``.
    """
    t1_star_col = t1_star[:, None]
    decay = torch.exp(-time_row / t1_star_col)  # (n_voxels, n_frames)
    # Normal-equation sums over frames (f: decay, y: signal, 1: ones).
    s_ff, s_f1, s_11 = (decay * decay).sum(-1), decay.sum(-1), float(n_frames)
    s_yf, s_y1 = (signal_flat * decay).sum(-1), signal_flat @ ones
    det = s_11 * s_ff - s_f1 * s_f1
    c1 = (s_y1 * s_ff - s_yf * s_f1) / det
    c2 = (s_yf * s_11 - s_y1 * s_f1) / det
    c1_col, c2_col = c1[:, None], c2[:, None]
    fit_const = c1_col * ones_row
    fit_decay = c2_col * decay
    resid = signal_flat - fit_const - fit_decay
    return (resid**2).sum(-1), c1, c2


# 1) Coarse grid search gives a bracket to refine and avoids local optima.
t1_star_grid = torch.logspace(
    np.log10(0.05), np.log10(6.0), 60, device=device, dtype=torch.float64
)
best_sse = torch.full((n_voxels,), float("inf"), dtype=torch.float64, device=device)
best_t1_star = torch.zeros(n_voxels, dtype=torch.float64, device=device)
for t1_star_trial in t1_star_grid:
    sse, _, _ = closed_form_sse(t1_star_trial.expand(n_voxels))
    better = sse < best_sse
    best_sse[better], best_t1_star[better] = sse[better], t1_star_trial

# 2) Golden-section refinement in log(T1*) space, bracketed around the grid
#    optimum. At every trial T1*, c1 and c2 are still solved in closed form
#    (VarPro); this replaces the fixed grid step with a continuous search.
log_step = (torch.log(t1_star_grid[1]) - torch.log(t1_star_grid[0])).item()
log_lo = torch.log(best_t1_star) - log_step
log_hi = torch.log(best_t1_star) + log_step
golden_ratio_conj = (5**0.5 - 1) / 2

probe_left = log_hi - golden_ratio_conj * (log_hi - log_lo)
probe_right = log_lo + golden_ratio_conj * (log_hi - log_lo)
sse_left, _, _ = closed_form_sse(torch.exp(probe_left))
sse_right, _, _ = closed_form_sse(torch.exp(probe_right))

# The bracket shrinks by 0.618^40 ~ 4e-9 in log-space, far below the noise floor.
n_golden_iters = 40
for _ in range(n_golden_iters):
    left_better = sse_left < sse_right
    log_hi = torch.where(left_better, probe_right, log_hi)
    probe_right = torch.where(left_better, probe_left, probe_right)
    sse_right = torch.where(left_better, sse_left, sse_right)
    probe_left = log_hi - golden_ratio_conj * (log_hi - log_lo)
    sse_left, _, _ = closed_form_sse(torch.exp(probe_left))

    right_better = ~left_better
    log_lo = torch.where(right_better, probe_left, log_lo)
    probe_left = torch.where(right_better, probe_right, probe_left)
    sse_left = torch.where(right_better, sse_right, sse_left)
    probe_right = log_lo + golden_ratio_conj * (log_hi - log_lo)
    sse_right, _, _ = closed_form_sse(torch.exp(probe_right))

t1_star = torch.exp((log_lo + log_hi) / 2)
_, c1_final, c2_final = closed_form_sse(t1_star)
s_steady, s_equilibrium = c1_final, -c2_final - c1_final

## Parameter maps

With flip angle $\alpha$:
$T_1 = T_1^*\,\frac{S_0}{S_{ss}}\cos\frac{\alpha}{2}$,
$T_2 = T_1^*\,\frac{\sin^2(\alpha/2)}{1 - (S_{ss}/S_0)\cos(\alpha/2)}$ and
proton density $= S_0 / \sin(\alpha/2)$. Voxels outside the brain mask are set
to zero.

In [ ]:
alpha_rad = np.deg2rad(flip_angle_deg)
cos_half_alpha, sin_half_alpha = np.cos(alpha_rad / 2), np.sin(alpha_rad / 2)

t1_map = t1_star * (s_equilibrium / s_steady) * cos_half_alpha
t2_map = t1_star * sin_half_alpha**2 / (1 - (s_steady / s_equilibrium) * cos_half_alpha)
pd_map = s_equilibrium / sin_half_alpha

t1_map = t1_map.reshape(n_x, n_x).cpu().numpy()
t2_map = t2_map.reshape(n_x, n_x).cpu().numpy()
pd_map = pd_map.reshape(n_x, n_x).cpu().numpy()
brain_2d = brain.reshape(n_x, n_x)  # collapse the trailing singleton dim

for parameter_map in (t1_map, t2_map, pd_map):
    parameter_map[~brain_2d] = 0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

im0 = axes[0].imshow(
    pd_map.T, cmap="gray", vmin=0, vmax=np.percentile(pd_map[brain_2d], 99)
)
axes[0].set_title("Proton Density")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(t1_map.T * 1e3, cmap="hot", vmin=0, vmax=3000)  # ms
axes[1].set_title("T_1 [ms]")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(t2_map.T * 1e3, cmap="hot", vmin=0, vmax=300)  # ms
axes[2].set_title("T_2 [ms]")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()